# 5. Limited dependent variables: hours worked (Tobit) and arrests (Poisson)

Two outcomes OLS handles badly: **hours** pile up at zero (corner solution), and **arrests** are
small counts. (Wooldridge Examples 17.2 and 17.3.)

In [1]:
import numpy as np
import statsmodels.formula.api as smf
import wooldridge as woo

from _setup import *  # noqa: F401,F403
import diagnostics as dg

## A. Tobit for annual hours of married women (MROZ)
325 of 753 women work zero hours. Question: how do young children affect labor supply — through
whether women work at all (extensive margin) or how much those who work do (intensive margin)?

In [2]:
mz = woo.data("mroz")
X = mz[["nwifeinc", "educ", "exper", "age", "kidslt6", "kidsge6"]].assign(expersq=mz["exper"] ** 2)
tb = dg.tobit(mz["hours"], X)


Tobit (censored below at 0.0; 325 below, 0 above, of 753), σ = 1122.0217, log-likelihood = -3819.09, converged = True
          latent_coef       se       z      p
const        965.3051 446.4322  2.1623 0.0306
nwifeinc      -8.8142   4.4591 -1.9767 0.0481
educ          80.6456  21.5832  3.7365 0.0002
exper        131.5643  17.2794  7.6140 0.0000
age          -54.4050   7.4185 -7.3337 0.0000
kidslt6     -894.0216 111.8779 -7.9910 0.0000
kidsge6      -16.2180  38.6402 -0.4197 0.6747
expersq       -1.8642   0.5377 -3.4672 0.0005

Average marginal effects (delta-method SEs):
            E[y|x]  se E[y|x]  E[y|uncensored,x]  se E[y|uncensored,x]  P(uncensored|x)  se P(uncensored|x)
nwifeinc   -5.1886     2.6214            -3.9688                2.0076          -0.0024              0.0012
educ       47.4731    12.6214            36.3123                9.7030           0.0222              0.0058
exper      77.4471     9.9976            59.2394                7.8337           0.0361          

Reading the margins for **kidslt6** (children under 6):
- `E[y|x]`: total effect on expected hours across all women.
- `E[y|uncensored,x]`: effect on hours among women who work.
- `P(uncensored|x)`: effect on the probability of working at all.

A young child reduces the probability of working by about 25 percentage points. The McDonald-Moffitt
decomposition (computed observation by observation, then averaged) splits the total effect on
expected hours between the two margins. The latent coefficient (−894 hours) is not an effect on
observed hours and should not be reported as one.

In [3]:
from scipy import stats

me = tb["marginal_effects"].loc["kidslt6"]
print(f"kidslt6: total {me['E[y|x]']:.0f} h, among workers {me['E[y|uncensored,x]']:.0f} h, "
      f"P(work) {me['P(uncensored|x)']:+.3f}")
b, s = tb["table"]["latent_coef"].values, tb["sigma"]
XX = np.column_stack([np.ones(len(X)), X.values])
c = XX @ b / s
P, lam = stats.norm.cdf(c), stats.norm.pdf(c) / stats.norm.cdf(c)
bj = tb["table"].loc["kidslt6", "latent_coef"]
intensive = np.mean(P * bj * (1 - lam * (c + lam)))                   # P(work) × change in hours of workers
extensive = np.mean((XX @ b + s * lam) * stats.norm.pdf(c) * bj / s)  # hours of workers × change in P(work)
print(f"McDonald-Moffitt: {intensive:.0f} h from fewer hours among workers + {extensive:.0f} h from leaving work "
      f"= {intensive + extensive:.0f} h ({extensive / (intensive + extensive):.0%} extensive)")

kidslt6: total -526 h, among workers -403 h, P(work) -0.246
McDonald-Moffitt: -273 h from fewer hours among workers + -253 h from leaving work = -526 h (48% extensive)


## B. Poisson regression for number of arrests in 1986 (CRIME1)

In [4]:
c = woo.data("crime1")
f = "narr86 ~ pcnv + avgsen + tottime + ptime86 + qemp86 + inc86 + black + hispan + born60"
pois = smf.poisson(f, c).fit(disp=0)
cd = dg.count_diagnostics(pois)


Poisson overdispersion: Pearson σ̂² = 1.517 (σ̂ = 1.232; 1 under Poisson)
                     alpha      t      p
NB2 (Var = μ + αμ²) 1.2457 4.8183 0.0000
NB1 (Var = (1+α)μ)  0.5001 4.1003 0.0000
Overdispersion: keep Poisson coefficients (QMLE) with robust SEs, or use quasi-Poisson SEs (× σ̂), or a negative binomial if count probabilities matter.
Zeros: observed 72.3% vs Poisson-implied 68.6%


Variance exceeds the mean (σ̂² ≈ 1.5), so the default Poisson standard errors are too small. The
coefficients remain consistent (quasi-MLE) if the mean is right: keep them, fix the errors.

In [5]:
robust = smf.poisson(f, c).fit(disp=0, cov_type="HC1")
b_pcnv = pois.params["pcnv"]
print(f"pcnv: coef {b_pcnv:.3f}; a 0.1 rise in the conviction probability → "
      f"{100*(1-np.exp(0.1*b_pcnv)):.1f}% fewer expected arrests")
print(f"SE of pcnv — Poisson {pois.bse['pcnv']:.3f}, robust {robust.bse['pcnv']:.3f}, "
      f"quasi-Poisson {cd['quasi_poisson_se']['pcnv']:.3f}")

pcnv: coef -0.402; a 0.1 rise in the conviction probability → 3.9% fewer expected arrests
SE of pcnv — Poisson 0.085, robust 0.101, quasi-Poisson 0.105


**Report.** A child under six lowers expected annual hours by about 530; the effect splits roughly
evenly between women leaving work (a 25-point drop in participation) and fewer hours among those who
keep working. The deterrence estimate (a higher past conviction rate is associated with fewer
arrests) survives the correction for overdispersion, with standard errors about 20% larger than
Poisson's.